# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a demonstration for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}\n")
print(f"Keywords: {getattr(metadata, 'keywords', None)}\n")
print(f"Published: {getattr(metadata,'datePublished', None)}\n")
print(f"License: {getattr(metadata,'license', None)}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as described by the schema.

We will list all `RecordSet` entities as well as their fields and columns using their `@id`s.

In [ ]:
# List all RecordSet objects and their fields/columns using @id
print("Available RecordSets (by @id):\n-------------------------------")
for record_set in dataset.metadata.record_sets:
    print(f"  RecordSet name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print(f"  Description: {getattr(record_set, 'description', '')}")
    print("    Fields (by @id):")
    for field in getattr(record_set, 'fields', []):
        print(f"      - {field.id}")
    print("    Columns (by @id):")
    for column in getattr(record_set, 'columns', []):
        print(f"      - {column.id}")
    print()

record_set_ids = [r.id for r in dataset.metadata.record_sets]
print("Summary of found record set @id's:\n", record_set_ids)

## 3. Data Extraction
Load data from a specific record set into pandas DataFrames for analysis. Entities are referenced by their `@id` as listed above.

We'll load all available record sets into DataFrames indexed by their `@id`.

In [ ]:
# Load each record set into a DataFrame, using the @id
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for the given record set
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
    else:
        dataframes[record_set_id] = pd.DataFrame()

# Print the columns of the first (main) record set
if len(record_set_ids) > 0:
    main_id = record_set_ids[0]
    print(f"Columns for main record set ({main_id}):\n", dataframes[main_id].columns.tolist())
    display(dataframes[main_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalizing numeric fields, and grouping.

Suppose our main quantitative column is "coverage_percent" (by `@id`), and we group by "sample" if available.

In [ ]:
# Identify candidate numeric and categorical fields by their @id or column name from previous output
# For illustration, we'll try to use 'coverage_percent' and 'sample' but you should adjust the field names if they're different.
main_record_set = main_id
df = dataframes[main_record_set]

# Try some candidate field IDs (replace with correct @id based on earlier overview if needed)
numeric_field_candidates = [c for c in df.columns if 'coverage' in c or 'percent' in c or 'abundance' in c or 'MW' in c or 'count' in c or 'peptide' in c]
print(f"Numeric field candidates in main record set: {numeric_field_candidates}")

if len(numeric_field_candidates) > 0:
    numeric_field = numeric_field_candidates[0]
else:
    numeric_field = df.columns[0]

# Attempt a numeric operation on the field
try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except Exception:
    pass

threshold = df[numeric_field].mean() if df[numeric_field].notna().sum() > 0 else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records in {main_record_set} with {numeric_field} > {threshold:.2f} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize the numeric field
if filtered_df[numeric_field].std() > 0:
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a likely categorical field (sample, accession, description, etc.)
group_field_candidates = [c for c in df.columns if 'sample' in c or 'type' in c or 'access' in c or 'desc' in c]
if group_field_candidates:
    group_field = group_field_candidates[0]
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped data by {group_field} (mean {numeric_field}):")
        display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of our main numeric field and, if grouped, visualize means by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field].dropna(), kde=True, bins=30, color="skyblue")
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If grouped, plot means by group
if 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
    plt.xticks(rotation=90)
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.tight_layout()
    plt.show()

## 6. Conclusion
We have loaded and explored the FAIR² dataset using [`mlcroissant`](https://github.com/mlcommons/croissant). We reviewed available record sets, loaded data by `@id`, filtered and visualized major quantitative fields, and demonstrated basic data analysis steps for large-scale proteomics data. Continue your investigation using the provided column `@id`s for more advanced analyses!